# 00 — Setup & Environment Configuration (Stage 0)

### What this notebook does:
1. **Mounts Google Drive** (if running in Google Colab) so your work and models are preserved.
2. **Verifies Python packages** (`gensim`, `pyyaml`, `requests`, `zstandard`, `tqdm`, `matplotlib`, `numpy`).
3. **Creates project directory structure** (`config/`, `manifests/`, `models/`, `vectors/`, `logs/`, `metadata/`).
4. **Validates configuration file** (`config/project_config.yaml`).

> **Data Analyst Note:** You do not need to change any code here. Just click **Runtime -> Run all** (or Shift+Enter through each cell).

In [ ]:
# Cell 1 — Mount Google Drive & Locate Project Root Directory
import os, sys
from pathlib import Path

# Check if running inside Google Colab
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_MOUNT = Path("/content/drive/MyDrive")
if IN_COLAB:
    if not os.path.isdir(str(DRIVE_MOUNT)):
        print("Mounting Google Drive...")
        drive.mount("/content/drive")

# A valid project root is a directory containing the config file (and src/).
def _has_root_marker(p: Path) -> bool:
    try:
        return p.is_dir() and (p / "config/project_config.yaml").is_file()
    except OSError:
        return False

def _find_project_root(start: Path):
    """Search upward, then a bounded depth downward, for the repo root."""
    for p in [start, *start.parents]:
        if _has_root_marker(p):
            return p
    for depth in (1, 2):
        for sub in start.glob("/".join(["*"] * depth)):
            if sub.is_dir() and _has_root_marker(sub):
                return sub
    return None

cwd = Path.cwd().resolve()
starts = [cwd] + ([DRIVE_MOUNT, Path("/content")] if IN_COLAB else [])

PROJECT_ROOT = None
seen = set()
for start in starts:
    found = _find_project_root(start)
    if found is not None:
        fp = found.resolve()
        if fp not in seen:
            seen.add(fp)
            PROJECT_ROOT = fp
            break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not auto-locate the project root (a folder containing "
        "'config/project_config.yaml' and 'src/').\n"
        "Fixes:\n"
        "  - In Colab, mount Drive, then %cd <project_root> before running this cell, or\n"
        "  - clone the repo into Google Drive, or\n"
        "  - run this notebook from inside the checkout.\n"
        "Alternative: set PROJECT_ROOT = Path('/absolute/path/to/repo') manually before this cell.\n"
    )

# Make shared utilities importable (e.g. `from src.storage import ...`).
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[STATUS] Running in Colab: {IN_COLAB}")
print(f"[STATUS] Project Root: {PROJECT_ROOT}")

In [ ]:
# Cell 2 — Verify and Install Required Dependencies
import importlib, subprocess

NEEDED_PACKAGES = {
    "yaml": "pyyaml",
    "requests": "requests",
    "zstandard": "zstandard",
    "tqdm": "tqdm",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "gensim": "gensim"
}

for module_name, pip_name in NEEDED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
        print(f"  [OK] {module_name}")
    except ImportError:
        print(f"  [INSTALLING] {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name])
        print(f"  [INSTALLED] {pip_name}")

print("All core dependencies are verified and ready!")

In [ ]:
# Cell 3 — Initialize Project Directory Skeleton & Load Configuration
import shutil
import yaml
from src.storage import atomic_write_text, sha256_file

ROOT = Path(PROJECT_ROOT)
SUBDIRECTORIES = [
    "config", "manifests", "manifests/pilot", "metadata/monthly_counts",
    "metadata/coverage_reports", "metadata/corpus_statistics", "shards/tokenized",
    "shards/tokenized/pilot", "shards/temporary", "shards/quarantine",
    "models/word2vec", "models/checkpoints", "models/fasttext", "vectors",
    "diagnostics/retrieval", "diagnostics/counts", "diagnostics/shards",
    "diagnostics/model_stability", "diagnostics/semantic_axes", "logs", "notebooks"
]

for d in SUBDIRECTORIES:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

CFG_PATH = ROOT / "config/project_config.yaml"
assert CFG_PATH.exists(), f"Missing configuration file at {CFG_PATH}."

cfg = yaml.safe_load(open(CFG_PATH, encoding="utf-8"))
CFG_SHA = sha256_file(CFG_PATH)

print(f"[CONFIG] Version: {cfg.get('config_version')} (SHA: {CFG_SHA[:12]}...)")
print(f"[CONFIG] Date Range: {cfg['corpus']['date_range']['start']} to {cfg['corpus']['date_range']['end']}")
print(f"[CONFIG] Word2Vec Embedding Dim: {cfg['embeddings']['dim']} | Window: {cfg['embeddings']['window']} | Epochs: {cfg['embeddings']['epochs']}")

In [ ]:
# Cell 4 — END-OF-SETUP SUMMARY
print("=" * 70)
print("STAGE 0 SETUP COMPLETE!")
print(f"  • Project Root: {ROOT}")
print(f"  • Config: {cfg['config_version']}")
print("  • Directory skeleton: Verified & Ready")
print("\nNext step: Open and run '01_test_data_access.ipynb' to verify data access.")
print("=" * 70)